### Welcome to Trident: A Python Package for Whole-Slide Image Processing 


This tutorial will guide you step-by-step to process a single whole-slide image (WSI) using Trident:

- Tissue vs. background segmentation 
- Tissue coordinate extraction
- Tissue feature extraction


In [12]:
import torch
import os
torch.cuda.is_available()

True

#### Inspection

In [13]:
# import openslide

# def inspect_wsi(wsi_path, show_thumbnail=False, thumb_size=(512, 512)):
#     """
#     Prints information about a whole slide image (WSI) using OpenSlide.

#     Args:
#         wsi_path (str): Path to the WSI file.
#         show_thumbnail (bool): Whether to show a thumbnail preview.
#         thumb_size (tuple): Thumbnail size, default (512, 512).

#     Returns:
#         slide: OpenSlide object (for further use)
#         thumb: PIL.Image thumbnail (if show_thumbnail=True), else None
#     """
#     slide = openslide.OpenSlide(wsi_path)
#     print("=== Whole Slide Image Information ===")
#     print(f"File:        {wsi_path}")
#     print(f"Dimensions:  {slide.dimensions}")
#     print(f"Levels:      {slide.level_count}")
#     print(f"Level dims:  {slide.level_dimensions}")
#     print(f"Level downsamples: {slide.level_downsamples}")

#     print("\n=== Properties ===")
#     for k, v in slide.properties.items():
#         print(f"{k}: {v}")

#     thumb = None
#     if show_thumbnail:
#         thumb = slide.get_thumbnail(thumb_size)
#         thumb.show()

#     return slide, thumb

#### Convert all .tiff to pyramidical structures

In [ ]:
import os 
import torch 
from PIL import Image
import geopandas as gpd
from IPython.display import display
from huggingface_hub import snapshot_download

from trident import OpenSlideWSI
from trident.segmentation_models import segmentation_model_factory
import h5py 

DEVICE = f"cuda:0" if torch.cuda.is_available() else "cpu"

encoder_list = ["uni_v1", "uni_v2", 
              "conch_v1", "conch_v15", 
              "virchow", "virchow2", 
              "phikon", "phikon_v2", 
              "gigapath", 
              "hoptimus0", "hoptimus1", 
              "musk", "midnight12k", 
              "kaiko-vits8", "kaiko-vits16", "kaiko-vitb8", "kaiko-vitb16", "kaiko-vitl14", 
              "lunit-vits8", 
              "hibou_l", 
              "ctranspath", 
              "resnet50"]

#### Segmentation and Extraction

In [ ]:
PATH = "../.."
# PATH = "storage"

local_wsi_dir = os.path.join(PATH, "data", "CAMELYON")

OUTPUT_DIR = local_wsi_dir

In [28]:
from trident.patch_encoder_models import encoder_factory

PATCH_ENCODER = "resnet50" # Visit the factory or check the README for a list of all available models
TARGET_MAG = 20
PATCH_SIZE = 512

# a. Instantiate model using the factory 
encoder = encoder_factory(PATCH_ENCODER)
encoder.eval()
encoder.to(DEVICE)

# Inspect h5 with patch coordinates
def print_attrs(name, obj):
    print(f"Object: {name}")
    for key, value in obj.attrs.items():
        print(f"  Attribute - {key}: {value}")

In [ ]:
from trident.patch_encoder_models import encoder_factory

PATCH_ENCODER = "resnet50" # Visit the factory or check the README for a list of all available models
TARGET_MAG = 20
PATCH_SIZE = 512

# a. Instantiate model using the factory 
encoder = encoder_factory(PATCH_ENCODER)
encoder.eval()
encoder.to(DEVICE)

for i, image in enumerate(images):
    print(f"Processing image {i+1} of {len(images)}: {image}")
    # b. Create OpenSlideWSI
    wsi_path = os.path.join(local_wsi_dir, image)
    slide = OpenSlideWSI(slide_path=wsi_path, lazy_init=False)

    # c. Run segmentation
    print("Running Segmentation...")
    segmentation_model = segmentation_model_factory("hest")
    geojson_contours = slide.segment_tissue(segmentation_model=segmentation_model, target_mag=10, job_dir=OUTPUT_DIR, device=DEVICE)

    # d. Visualize contours
    contour_image = Image.open(os.path.join(OUTPUT_DIR, 'contours', image.replace('.tif', '.jpg')))
    gdf = gpd.read_file(geojson_contours)
    gdf.head(n=10)

    # Run patch coordinate extraction
    print("Extracting Patch Coordinates...")
    coords_path = slide.extract_tissue_coords(
        target_mag=TARGET_MAG,
        patch_size=PATCH_SIZE,
        save_coords=OUTPUT_DIR
    )
    # Visualize
    viz_coords_path = slide.visualize_coords(
        coords_path=coords_path,
        save_patch_viz=os.path.join(OUTPUT_DIR, "visualization")
    )

    features_dir = os.path.join(OUTPUT_DIR, f"features_{PATCH_ENCODER}")
    feats_path = slide.extract_patch_features(
        patch_encoder=encoder,
        coords_path=coords_path,
        save_features=features_dir,
        device=DEVICE
    )

In [ ]:
import os
from pathlib import Path
import torch
from trident import OpenSlideWSI
from trident.segmentation_models import segmentation_model_factory
from trident.patch_encoder_models import encoder_factory

def extract_wsi_patch_features(
    encoder_name: str,
    target_mag: int,
    patch_size: int,
    wsi_dir: str,
    images: list[str],
    output_dir: str,
    device: torch.device = None
) -> None:
    """
    Extract tissue patches and their feature embeddings from a list of WSIs.

    Args:
        encoder_name: Name of the patch encoder (e.g. "resnet50").
        target_mag:   Magnification at which to extract patches (e.g. 20).
        patch_size:   Size of each square patch in pixels (e.g. 512).
        wsi_dir:      Directory containing your WSI files.
        images:       List of filenames (with extension) of WSIs to process.
        output_dir:   Base directory where outputs (contours, coords, features) will be saved.
        device:       torch.device to use (e.g. torch.device("cuda:0")); 
                      if None, automatically selects cuda if available.
    """
    # pick device
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # instantiate and prepare the encoder
    encoder = encoder_factory(encoder_name)
    encoder.eval().to(device)

    # ensure output directory exists
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    for idx, fname in enumerate(images, start=1):
        print(f"[{encoder_name}] ({idx}/{len(images)}) Processing: {fname}")
        wsi_path = os.path.join(wsi_dir, fname)
        slide = OpenSlideWSI(slide_path=wsi_path, lazy_init=False)

        # 1) Tissue segmentation
        print("  - segmenting tissue...")
        seg_model = segmentation_model_factory("hest")
        geojson = slide.segment_tissue(
            segmentation_model=seg_model,
            target_mag=10,
            job_dir=output_dir,
            device=device
        )

        # 2) Patch coordinate extraction
        print("  - extracting patch coordinates...")
        coords_file = slide.extract_tissue_coords(
            target_mag=target_mag,
            patch_size=patch_size,
            save_coords=output_dir
        )

        # 3) Feature extraction
        print("  - extracting patch features...")
        feats_dir = Path(output_dir) / f"features_{encoder_name}"
        feats_dir.mkdir(parents=True, exist_ok=True)
        feats_file = slide.extract_patch_features(
            patch_encoder=encoder,
            coords_path=coords_file,
            save_features=str(feats_dir),
            device=device
        )

        print(f"    → features saved to {feats_file}\n")

In [29]:
test_images = []
train_images = []

test_dir = os.path.join(local_wsi_dir, "test")
train_dir = os.path.join(local_wsi_dir, "train")

import os
for file in os.listdir(test_dir):
    if file.endswith(".tif"):
        test_images.append(file)

for file in os.listdir(train_dir):
    if file.endswith(".tif"):
        train_images.append(file)

In [ ]:
for enc in encoder_list:
    extract_wsi_patch_features(
        encoder_name=enc,
        target_mag=20,
        patch_size=512,
        wsi_dir=test_dir,
        images=test_images,
        output_dir=os.path.join(test_dir, "feats")
    )

for enc in encoder_list:
    extract_wsi_patch_features(
        encoder_name=enc,
        target_mag=20,
        patch_size=512,
        wsi_dir=train_dir,
        images=train_images,
        output_dir=os.path.join(train_dir, "feats")
    )

['test_001.tif', 'test_002.tif', 'test_003.tif', 'test_004.tif']

In [ ]:
# # local_wsi_dir = r"C:\Users\Dhyey\Documents\GitHub\trident\tutorials\tutorial-1\wsis"
# # os.makedirs(os.path.join(base_dir, "pyramidal"), exist_ok=True)
# # local_wsi_dir = r"Data\MIDOGpp_pyr"

# for i, image in enumerate(images):
#     print(f"Processing image {i+1} of {len(images)}: {image}")
#     # b. Create OpenSlideWSI
#     wsi_path = os.path.join(local_wsi_dir, image)
#     slide = OpenSlideWSI(slide_path=wsi_path, lazy_init=False)

#     # c. Run segmentation
#     print("Running Segmentation...")
#     segmentation_model = segmentation_model_factory("hest")
#     geojson_contours = slide.segment_tissue(segmentation_model=segmentation_model, target_mag=10, job_dir=OUTPUT_DIR, device=DEVICE)

#     # d. Visualize contours
#     contour_image = Image.open(os.path.join(OUTPUT_DIR, 'contours', image.replace('.tif', '.jpg')))
#     # display(contour_image)

#     # e. Check contours saved into GeoJSON with GeoPandas
#     gdf = gpd.read_file(geojson_contours)
#     gdf.head(n=10)


#     # Run patch coordinate extraction
#     print("Extracting Patch Coordinates...")
#     coords_path = slide.extract_tissue_coords(
#         target_mag=TARGET_MAG,
#         patch_size=PATCH_SIZE,
#         save_coords=OUTPUT_DIR
#     )
#     # Visualize
#     viz_coords_path = slide.visualize_coords(
#         coords_path=coords_path,
#         save_patch_viz=os.path.join(OUTPUT_DIR, "visualization")
#     )
#     # display(Image.open(viz_coords_path))
 
#     # with h5py.File(coords_path, 'r') as h5_file:
#     #     print("Contents and Attributes in patch coords file:")
#     #     h5_file.visititems(print_attrs)

#     # Run feature extraction
#     print("Extracting Patch Features...")
#     features_dir = os.path.join(OUTPUT_DIR, f"features_{PATCH_ENCODER}")
#     feats_path = slide.extract_patch_features(
#         patch_encoder=encoder,
#         coords_path=coords_path,
#         save_features=features_dir,
#         device=DEVICE
#     )

#     # Inspect h5 with patch features 
#     # with h5py.File(feats_path, 'r') as h5_file:
#     #     print("Contents and Attributes in feats file:")
#     #     h5_file.visititems(print_attrs)

In [2]:
import h5py

def inspect_h5(h5_path, print_values=False, max_items=5):
    print(f"Inspecting: {h5_path}")
    with h5py.File(h5_path, 'r') as f:
        def print_group(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  [Dataset] {name}: shape={obj.shape}, dtype={obj.dtype}")
                if obj.attrs:
                    print(f"    Attributes:")
                    for k, v in obj.attrs.items():
                        print(f"      {k}: {v}")
                if print_values:
                    try:
                        vals = obj[()]
                        print(f"    Sample values: {vals.ravel()[:max_items]}")
                    except Exception as e:
                        print(f"    Could not print values: {e}")
            elif isinstance(obj, h5py.Group):
                print(f"[Group] {name}/")
        f.visititems(print_group)

# Example usage
h5_path = r"C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON\test\feats\features_resnet50\test_001.h5"
inspect_h5(h5_path, print_values=True)

Inspecting: C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON\test\feats\features_resnet50\test_001.h5
  [Dataset] coords: shape=(4300, 2), dtype=int64
    Attributes:
      level0_magnification: 40
      name: test_001
      overlap: 0
      patch_size: 512
      patch_size_level0: 1024
      savetodir: data\CAMELYON\test\feats
      target_magnification: 20
    Sample values: [ 1024 34816  1024 35840  1024]
  [Dataset] features: shape=(4300, 1024), dtype=float32
    Attributes:
      name: test_001
      savetodir: data\CAMELYON\test\feats\features_resnet50
    Sample values: [0.10259934 0.01180088 0.01838106 0.00494716 0.01543034]


In [ ]:
# import h5py

# h5_path = r"C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON\features_resnet50\meta_010.h5"

# with h5py.File(h5_path, "r") as f:
#     # List all top-level datasets and their shapes
#     print("Datasets in file:", list(f.keys()))

#     # --- Coordinates ---
#     coords = f["coords"][()]  # (N, 2) array of spatial patch coordinates
#     print("coords shape:", coords.shape)
#     print("First 5 coords:\n", coords[:5])

#     # Attributes for coords
#     for k, v in f["coords"].attrs.items():
#         print(f"coords attribute: {k} = {v}")

#     # --- Features ---
#     features = f["features"][()]  # (N, 1024) feature embedding vectors
#     print("features shape:", features.shape)
#     print("First 2 feature embeddings:\n", features[:2])

#     # Attributes for features
#     for k, v in f["features"].attrs.items():
#         print(f"features attribute: {k} = {v}")

#     # Access a single feature vector for patch 0
#     print("First feature vector (patch 0):", features[0])